In [4]:
import pandas as pd

In [5]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

In [6]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.shape

(5572, 2)

In [8]:
df.columns

Index(['label', 'message'], dtype='str')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [10]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [11]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(403)

In [13]:
import sys
print(sys.executable)

c:\Internship_Task\Spam_Mail_Detector\.venv\Scripts\python.exe


In [14]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print(len(stop_words))
print(list(stop_words)[:10])

198
['its', 'couldn', 'myself', "aren't", 'ours', 'more', 'for', 'other', "she's", "we'd"]


In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [16]:
df["message"].iloc[0].lower()    # First row (by position)

'go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat...'

# Tokenization

In [17]:
def preprocess_text(text):
    text = text.lower()
    
    tokens = word_tokenize(text)
    
    tokens = [word for word in tokens if word not in stop_words]
    
    return tokens

In [18]:
df["processed_message"] = df["message"].apply(preprocess_text)

In [19]:
sample = "Congratulations! You have won a free prize."
preprocess_text(sample)

['congratulations', '!', 'free', 'prize', '.']

In [20]:
df["processed_message"] = df["message"].apply(preprocess_text)

In [21]:
df[["message", "processed_message"]].head()

,message,processed_message
0,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, ,, crazy, .., available, b..."
1,Ok lar... Joking wif u oni...,"[ok, lar, ..., joking, wif, u, oni, ...]"
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, ..., u, c, already, ..."
4,"Nah I don't think he goes to usf, he lives aro...","[nah, n't, think, goes, usf, ,, lives, around,..."


## Feature Extraction using TF-IDF

Machine learning models cannot directly process text.
Therefore, the preprocessed messages need to be converted into numerical features.

TF-IDF (Term Frequency-Inverse Document Frequency) assigns a numerical importance
score to words based on how frequently they occur in a message and how common
they are across all messages.

The TF-IDF representation will be used as input features for our classification model.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
tfidf = TfidfVectorizer()

In [24]:
df["processed_text"] = df["processed_message"].apply(
    lambda tokens: " ".join(tokens)
)

In [25]:
df[["message", "processed_message", "processed_text"]].head()

,message,processed_message,processed_text
0,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, ,, crazy, .., available, b...","go jurong point , crazy .. available bugis n g..."
1,Ok lar... Joking wif u oni...,"[ok, lar, ..., joking, wif, u, oni, ...]",ok lar ... joking wif u oni ...
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin...",free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, ..., u, c, already, ...",u dun say early hor ... u c already say ...
4,"Nah I don't think he goes to usf, he lives aro...","[nah, n't, think, goes, usf, ,, lives, around,...","nah n't think goes usf , lives around though"


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

In [27]:
X = tfidf.fit_transform(df["processed_text"])

In [28]:
X.shape

(5572, 8644)

In [29]:
feature_names = tfidf.get_feature_names_out()

print("Feature matrix shape:", X.shape)
print("Number of features:", len(feature_names))
print("First 20 features:", feature_names[:20])

Feature matrix shape: (5572, 8644)
Number of features: 8644
First 20 features: ['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '0125698789' '02' '0207' '02072069400' '02073162414'
 '02085076972' '021' '03' '04' '0430' '05' '050703']


In [30]:
y = df["label"]

In [31]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (5572, 8644)
Target shape: (5572,)


## Train-Test Split

The dataset is divided into training and testing sets.

- Training data is used to train the machine learning model.
- Testing data is used to evaluate the model on unseen messages.

We use an 80:20 split, where 80% of the data is used for training and 20% for testing.

In [32]:
from sklearn.model_selection import train_test_split

In [33]:
X_text = df["processed_text"]
y = df["label"]

In [34]:
print("Text samples:", len(X_text))
print("Labels:", len(y))

Text samples: 5572
Labels: 5572


In [35]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [36]:
print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

Training samples: 4457
Testing samples: 1115


In [37]:
print("Training labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training labels:
label
ham     3859
spam     598
Name: count, dtype: int64

Testing labels:
label
ham     966
spam    149
Name: count, dtype: int64


In [38]:
tfidf = TfidfVectorizer()

In [39]:
X_train = tfidf.fit_transform(X_train_text)

In [40]:
X_test = tfidf.transform(X_test_text)

In [41]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (4457, 7600)
X_test shape: (1115, 7600)


## Train-Test Split Result

The dataset was divided into training and testing sets using an 80:20 ratio.

TF-IDF was fitted only on the training data and then used to transform both
the training and testing data. This prevents information from the test set
from influencing the feature extraction process and helps avoid data leakage.

In [42]:
print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Training samples: 4457
Testing samples: 1115
X_train shape: (4457, 7600)
X_test shape: (1115, 7600)


## Train the Spam Classifier

In this step, we train a Multinomial Naive Bayes classifier using the TF-IDF
features generated from the SMS messages.

Multinomial Naive Bayes is a simple and effective machine learning algorithm
for text classification problems such as spam detection.

In [43]:
from sklearn.naive_bayes import MultinomialNB

In [44]:
model = MultinomialNB()

In [45]:
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7600)","[[0. ,0. ,0.24,...,0.19,0. ,0.34], [2.28,4.87,0. ,...,0. ,0.22,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7600)","[[-9.76,-9.76,-9.54,...,-9.58,-9.76,-9.46], [-8.02,-7.43,-9.2 ,...,-9.2 ,-9. ,-9.2 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7600


In [46]:
y_pred = model.predict(X_test)

In [47]:
print(y_pred[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [48]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(20)

,Actual,Predicted
0,ham,ham
1,ham,ham
2,ham,ham
3,spam,spam
4,ham,ham
5,ham,ham
6,ham,ham
7,ham,ham
8,ham,ham
9,ham,ham


In [49]:
test_message = ["Hey, are we still meeting today?"]

test_vector = tfidf.transform(test_message)

prediction = model.predict(test_vector)

print("Prediction:", prediction[0])

Prediction: ham


In [50]:
test_message = ["Congratulations! You have won a free prize. Call now to claim!"]

test_vector = tfidf.transform(test_message)

prediction = model.predict(test_vector)

print("Prediction:", prediction[0])

Prediction: spam


## Model Training Result

A Multinomial Naive Bayes classifier was trained using the TF-IDF features from
the training dataset.

The trained model was then used to predict whether previously unseen SMS
messages were spam or ham.

The predictions will be evaluated using accuracy, precision, recall and F1-score
in the next step.